# CIFAR residual CNN competition — Colab Pro

**4,480,474 trainable parameters.** Train from scratch using only your competition archive.
This notebook compares validation runs, trains the selected recipe on all allowed images,
and writes a checked `submission.csv`. No Kaggle score is available before training/evaluation.

**Before running:** select a GPU runtime (A100 or L4 if available), create
`MyDrive/cifar_resnet_project`, and upload `data.zip` into it. Code is cloned from this public repository automatically;
you may alternatively supply the original `cifar-resnet-colab.zip`.
Keep both as ZIP files. See the included README for the methods, citations, and rules.

Checkpoints are saved to Drive every completed epoch. After a disconnect, reconnect a GPU
runtime and rerun this notebook from the start; completed runs are skipped and interrupted
runs resume. Keep the project folder, configurations, and deterministic setting unchanged.
The current unfinished epoch may need to repeat. Run cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, time

PROJECT = Path('/content/drive/MyDrive/cifar_resnet_project')
DATA_ZIP = PROJECT / 'data.zip'
REPO_ZIP = PROJECT / 'cifar-resnet-colab.zip'
RUNS = PROJECT / 'runs'
RUN_COMPARISON = True    # False: main recipe only, saves GPU time
DETERMINISTIC = False   # Set True before first run for stricter replay, at some speed cost
WORKERS = 4             # Use 2 if the runtime has few CPU cores
RUNS.mkdir(parents=True, exist_ok=True)
assert DATA_ZIP.is_file(), f'Upload your original data.zip to {PROJECT}'
REPO_URL = 'https://github.com/mj3664-web/cifar-resnet-colab.git'

## Load the repository and check the GPU

The code runs from local runtime storage for speed. Only durable artifacts and original ZIPs
stay on Drive. Colab's installed PyTorch/torchvision CUDA pair is retained; the notebook
records its exact versions. No model weights or datasets are downloaded.

In [ ]:
LOCAL = Path('/content/cifar_competition')
LOCAL.mkdir(exist_ok=True)
REPO = LOCAL / 'cifar-resnet-colab'
if REPO_ZIP.is_file():
    with zipfile.ZipFile(REPO_ZIP) as archive:
        for entry in archive.infolist():
            destination = (LOCAL / entry.filename).resolve()
            assert destination.is_relative_to(LOCAL.resolve()), 'Unsafe repository ZIP path'
        archive.extractall(LOCAL)
elif not (REPO / 'src/train.py').is_file():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
if (REPO / '.git').is_dir():
    code_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    saved_commit = RUNS / 'code_commit.txt'
    if saved_commit.exists():
        subprocess.run(['git', '-C', str(REPO), 'checkout', saved_commit.read_text().strip()], check=True)
    else:
        saved_commit.write_text(code_commit)
    print('Training code commit:', saved_commit.read_text())
assert (REPO / 'src/train.py').is_file(), 'Unexpected repository ZIP structure'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch, torchvision, numpy as np
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('CUDA:', torch.version.cuda)

def run_module(module, *args):
    command = [sys.executable, '-u', '-m', module, *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

def train(config, output, full_data=False, stop_after=None):
    args = ['--data', CACHE, '--config', config, '--out', output,
            '--workers', WORKERS, '--resume']
    if full_data:
        args += ['--full-data']
    if DETERMINISTIC:
        args += ['--deterministic']
    if stop_after is not None:
        args += ['--stop-after', stop_after]
    run_module('src.train', *args)

## Run correctness checks and prepare the supplied dataset

The integration test uses tiny synthetic fixtures, solely to check the software. These
fixtures are never used in competition training. The real dataset is validated separately.
Preparation checks test-image structure but learns no statistics or targets from test images.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
CACHE = LOCAL / 'cache'
local_zip = LOCAL / 'data.zip'
shutil.copy2(DATA_ZIP, local_zip)
run_module('src.data', '--zip', local_zip, '--out', CACHE)
manifest = json.loads((CACHE / 'manifest.json').read_text())
assert manifest['train_count'] == 50000 and manifest['test_count'] == 2000
print(manifest)

from src.model import build_model
config = json.loads((REPO / 'configs/main.json').read_text())
model = build_model(config)
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
del model

## Start the main experiment and estimate runtime

This first segment completes three epochs of the **original 400-epoch schedule**. It is
real training and continues in the next cell. The first epoch can be slower due to warmup.
The epoch time includes validation of raw and EMA weights using horizontal-flip averaging.
Use this measured time to decide whether to keep `RUN_COMPARISON = True`.

In [ ]:
MAIN_RUN = RUNS / 'main'
train(REPO / 'configs/main.json', MAIN_RUN, stop_after=3)
history = [json.loads(line) for line in (MAIN_RUN / 'history.jsonl').read_text().splitlines()]
recent = history[-min(3, len(history)):]
seconds = sum(row['seconds'] for row in recent) / len(recent)
print(f'Measured mean epoch time: {seconds:.1f} seconds')
print(f'Rough main-run GPU time for 400 epochs: {seconds * 400 / 3600:.1f} hours')
print('Add a second full training budget for the final fit, and 300 epochs if comparing.')

## Complete development training

Validation is a fixed stratified 5,000-image holdout from the allowed training data.
Each completed epoch saves `last.pt`; improvements save a single-network `best.pt`.
An incomplete run can be resumed without changing the configuration. Output directories
must be unique for genuinely different experiments.

This cell may take many hours. If the session ends, rerun the notebook after reconnecting.
Do not reduce `epochs` to resume sooner: that would change the cosine schedule.

In [ ]:
train(REPO / 'configs/main.json', MAIN_RUN)
CANDIDATES = [MAIN_RUN]
if RUN_COMPARISON:
    BASELINE_RUN = RUNS / 'baseline'
    train(REPO / 'configs/baseline.json', BASELINE_RUN)
    CANDIDATES.append(BASELINE_RUN)

## Inspect validation results and select the recipe

Selection uses only held-out training labels. A close difference may be seed noise; for a
more rigorous comparison, add another seed in a separate experiment before final fitting.
The script requires completed runs on identical validation splits.

In [ ]:
SELECTED_CONFIG = RUNS / 'selected.json'
run_module('src.select', '--runs', *CANDIDATES, '--out', SELECTED_CONFIG)
selection = json.loads(SELECTED_CONFIG.with_suffix('.selection.json').read_text())
print(json.dumps(selection, indent=2))
run_module('src.evaluate', '--data', CACHE,
           '--checkpoint', Path(selection['selected_run']) / 'best.pt', '--workers', WORKERS)

In [ ]:
# Optional visual inspection. Matplotlib is included in typical Colab runtimes.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for candidate in CANDIDATES:
    rows = [json.loads(s) for s in (candidate / 'history.jsonl').read_text().splitlines()]
    axes[0].plot([r['epoch'] for r in rows], [r['train_loss'] for r in rows], label=candidate.name)
    axes[1].plot([r['epoch'] for r in rows], [100*r['ema_accuracy'] for r in rows], label=candidate.name+' EMA')
    axes[1].plot([r['epoch'] for r in rows], [100*r['raw_accuracy'] for r in rows], alpha=.5, label=candidate.name+' raw')
axes[0].set(xlabel='Epoch', ylabel='Augmented training loss')
axes[1].set(xlabel='Epoch', ylabel='Validation accuracy (%)')
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend()
plt.tight_layout()
plt.savefig(RUNS / 'development_curves.png', dpi=160)
plt.show()

## Train the final model on all 50,000 allowed images

This is a new fit from random initialization with the selected recipe, raw/EMA choice,
and full epoch budget. It does not select a checkpoint against the old validation set,
because those images are now part of training. Normalization is refit on all allowed
training images. The final exported checkpoint contains one network under 5M parameters.

If you later select a different recipe, use a new `FINAL_RUN` directory.

In [ ]:
FINAL_RUN = RUNS / 'final'
train(SELECTED_CONFIG, FINAL_RUN, full_data=True)
assert (FINAL_RUN / 'final.pt').is_file(), 'Final fit must complete before submission generation'
print(json.loads((FINAL_RUN / 'metadata.json').read_text()))

## Generate and download the submission

Prediction uses the saved normalization and inference settings. IDs are joined to the
sample submission's exact order and labels are checked to be integers 0–9. Submit the CSV
manually to the official competition page. Never use hidden evaluation data for local
training, validation, or selection. Keep the final checkpoint to reproduce these predictions.

In [ ]:
SUBMISSION = PROJECT / 'submission.csv'
run_module('src.predict', '--data', CACHE, '--checkpoint', FINAL_RUN / 'final.pt',
           '--out', SUBMISSION, '--workers', WORKERS)
print(SUBMISSION.read_text().splitlines()[:6])
print(json.loads(SUBMISSION.with_suffix('.metadata.json').read_text()))
from google.colab import files
files.download(str(SUBMISSION))

## Save the reproduction record and complete the GitHub hand-in

The source ZIP already contains the model, training, preprocessing, configurations,
tests, citations, and README. Upload its source files to your team's GitHub repository.
Do not commit the competition dataset. Fill in `RESULTS.md` with actual metrics, the
selected configuration, code commit, final checkpoint location/hash, and official score.
The JSON record below captures locally measured results; it makes no Kaggle score claim.

In [ ]:
record = {
    'development_selection': selection,
    'final_training': json.loads((FINAL_RUN / 'metadata.json').read_text()),
    'submission': json.loads(SUBMISSION.with_suffix('.metadata.json').read_text()),
    'official_kaggle_score': None,
    'repository_url': None,
    'checkpoint_path': str(FINAL_RUN / 'final.pt')
}
(PROJECT / 'reproduction_record.json').write_text(json.dumps(record, indent=2))
print('Saved:', PROJECT / 'reproduction_record.json')
print('Keep the original code ZIP, dataset ZIP, selected config, environment files, and final.pt.')